In [ ]:
from google.colab import files
uploaded = files.upload()

Saving study_prompt_answered.csv to study_prompt_answered.csv


In [ ]:
import pandas as pd
moods=pd.read_csv("study_prompt_answered.csv")
moods.head()

,pid,EventName,Time,Time_utc,CampaignProgressionAmount,CurrentGameMode,CurrentJobName,CurrentSessionLength,LastSubtaskCompleted,LastTaskCompleted,LevelProgressionAmount,LastStudyPromptType,response
0,p1,study_prompt_answered,2022-08-18 18:38:31,2022-08-18 23:38:31,NaN,NaN,NaN,0,NaN,NaN,NaN,Wellbeing,653.0
1,p1,study_prompt_answered,2022-08-18 18:45:01,2022-08-18 23:45:01,0.026316,Career,RESIDENTIALSMALL_BACKYARD,6,WASH_Backyard.WASH_Backyard.WASH_Environment.P...,WASH_PWVan,0.168267,Wellbeing,817.0
2,p1,study_prompt_answered,2022-08-18 18:51:22,2022-08-18 23:51:22,0.026316,Career,RESIDENTIALSMALL_BACKYARD,13,WASH_Backyard.WASH_Backyard.WASH_Environment.F...,WASH_PWVan,0.429364,Wellbeing,810.0
3,p1,study_prompt_answered,2022-08-18 19:02:01,2022-08-19 00:02:01,0.026316,Career,RESIDENTIALSMALL_BACKYARD,7,WASH_Backyard.WASH_Backyard.WASH_Environment.F...,NaN,0.726184,Focus,790.0
4,p1,study_prompt_answered,2022-08-18 19:08:47,2022-08-19 00:08:47,0.026316,Career,RESIDENTIALSMALL_BACKYARD,13,WASH_Backyard.WASH_Backyard.WASH_SwingSofa.Swi...,NaN,0.834437,Focus,794.0


In [ ]:
moods.isna().sum()

,0
pid,0
EventName,0
Time,0
Time_utc,0
CampaignProgressionAmount,24240
CurrentGameMode,24240
CurrentJobName,24240
CurrentSessionLength,0
LastSubtaskCompleted,30462
LastTaskCompleted,345185


In [ ]:
moods = moods.drop('Time_utc', axis=1)

In [ ]:
moods = moods.drop('Time', axis=1)

In [ ]:
moods = moods.drop('LastTaskCompleted', axis=1)

In [ ]:
moods = moods.drop('LastSubtaskCompleted', axis=1)

In [ ]:
moods = moods.drop('EventName', axis=1)

In [ ]:
moods = moods.drop('response', axis=1)

In [ ]:
moods = moods.drop('pid', axis=1)

In [ ]:
moods = moods.drop('CurrentJobName', axis=1)

In [ ]:
moods = moods.drop('LevelProgressionAmount', axis=1)

In [ ]:
moods = moods.drop('CurrentGameMode', axis=1)

In [ ]:
moods = moods.drop('CampaignProgressionAmount', axis=1)

In [ ]:
moods.head()

,CurrentSessionLength,LastStudyPromptType
0,0,Wellbeing
1,6,Wellbeing
2,13,Wellbeing
3,7,Focus
4,13,Focus


In [ ]:
print("Unique values in LastStudyPromptType:")
print(moods['LastStudyPromptType'].unique())

Unique values in LastStudyPromptType:
['Wellbeing' 'Focus' 'Enjoyment' 'Competence' 'Autonomy' 'Immersion']


In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
moods['LastStudyPromptType']=le.fit_transform(moods.LastStudyPromptType.values)
moods.head()

,CurrentSessionLength,LastStudyPromptType
0,0,5
1,6,5
2,13,5
3,7,3
4,13,3


In [ ]:
le = LabelEncoder()
le.fit(moods['LastStudyPromptType'])
print(list(le.classes_))

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
moods_fix = pd.DataFrame(imputer.fit_transform(moods), columns=moods.columns)

In [ ]:
moods_fix.isna().sum()

,0
CurrentSessionLength,0
LastStudyPromptType,0


In [ ]:
moods_fix['LastStudyPromptType'].value_counts()

,count
LastStudyPromptType,
5.0,177802
2.0,154152
3.0,153453
0.0,72529
4.0,72298
1.0,71975


In [ ]:
x = moods_fix.drop('LastStudyPromptType', axis=1)
y = moods_fix.LastStudyPromptType

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
x_scaled = scaler.fit_transform(x)

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.3, random_state=0, stratify=y)

In [ ]:
group_map = {
    0: 0,
    1: 0,
    2: 2,
    3: 1,
    4: 1,
    5: 2
}

In [ ]:
y_train_grouped = y_train.replace(group_map)
y_test_grouped = y_test.replace(group_map)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0, sampling_strategy='not majority')
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train_grouped)

print("Class distribution after SMOTE:", pd.Series(y_train_resampled).value_counts())

Class distribution after SMOTE: LastStudyPromptType
2.0    232367
1.0    232367
0.0    232367
Name: count, dtype: int64


In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=0
)
model.fit(x_train_resampled, y_train_resampled)
predict = model.predict(x_test)

from sklearn.metrics import accuracy_score
print("Decision Tree Accuracy:", accuracy_score(y_test_grouped, predict))

Decision Tree Accuracy: 0.2959086313211148


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model=KNeighborsClassifier(n_neighbors=50)
model.fit(x_train_resampled, y_train_resampled)
predict=model.predict(x_test)
from sklearn.metrics import accuracy_score
print('KNN Classifier:', accuracy_score(y_test_grouped,predict))

KNN Classifier: 0.44424507388577966


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test_grouped, predict))

              precision    recall  f1-score   support

         0.0       0.22      0.02      0.04     43351
         1.0       0.33      0.21      0.26     67725
         2.0       0.48      0.79      0.60     99587

    accuracy                           0.44    210663
   macro avg       0.34      0.34      0.30    210663
weighted avg       0.38      0.44      0.37    210663



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=0,
    n_jobs=-1
)
rf_model.fit(x_train_resampled, y_train_resampled)

rf_pred = rf_model.predict(x_test)

print("Random Forest Accuracy:", accuracy_score(y_test_grouped, rf_pred))

Random Forest Accuracy: 0.2970953608369766
